# ✅ Notebook 5: Validation & SPARQL Querying

Validate ontology consistency with **OWL reasoners** and query with **SPARQL**.

---

In [ ]:
!pip install -q owlready2 rdflib networkx matplotlib
!git clone https://github.com/YOUR_USERNAME/ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'ontology-builder')

## 1. Build a Test Ontology

Create an ontology to validate (or load from previous notebook).

In [ ]:
import os
from src.ontology_builder import OntologyBuilder
from src.concept_extractor import Concept
from src.relation_extractor import Relation

os.makedirs('output', exist_ok=True)

builder = OntologyBuilder(iri='http://example.org/test-ontology')

builder.add_concepts([
    Concept(name='Animal', label='Animal'),
    Concept(name='Mammal', label='Mammal', parent='Animal'),
    Concept(name='Bird', label='Bird', parent='Animal'),
    Concept(name='Dog', label='Dog', parent='Mammal'),
    Concept(name='Cat', label='Cat', parent='Mammal'),
    Concept(name='Eagle', label='Eagle', parent='Bird'),
    Concept(name='Habitat', label='Habitat'),
    Concept(name='Forest', label='Forest', parent='Habitat'),
    Concept(name='Ocean', label='Ocean', parent='Habitat'),
])

builder.add_relations([
    Relation(subject='Animal', predicate='livesIn', object='Habitat'),
    Relation(subject='Dog', predicate='eats', object='Mammal'),
    Relation(subject='Eagle', predicate='hunts', object='Animal'),
])

builder.add_data_properties([
    {'concept': 'Animal', 'attribute': 'hasCommonName', 'value_type': 'string'},
    {'concept': 'Animal', 'attribute': 'hasLifespanYears', 'value_type': 'integer'},
])

builder.save('output/test_ontology.owl')
print('Ontology saved. Stats:', builder.get_stats())

## 2. Validate with Reasoner

In [ ]:
from src.validator import OntologyValidator

validator = OntologyValidator(reasoner='hermit')
report = validator.validate(builder.onto)

print(report.summary())
print(f'\nStats: {report.stats}')

if report.inferred_subsumptions:
    print(f'\nInferred Subsumptions ({len(report.inferred_subsumptions)}):')
    for child, parent in report.inferred_subsumptions[:15]:
        print(f'  {child} ⊑ {parent}')

## 3. SPARQL Queries

In [ ]:
from src.query_engine import QueryEngine

engine = QueryEngine('output/test_ontology.owl')
print(f'Loaded {engine.count_triples()} triples\n')

# Query 1: All classes
print('=== All OWL Classes ===')
classes = engine.get_all_classes()
for c in classes:
    name = c['class'].split('/')[-1].split('#')[-1]
    label = c.get('label', '')
    print(f'  {name:<20} label: {label}')

In [ ]:
# Query 2: Class hierarchy
print('=== Class Hierarchy ===')
hierarchy = engine.get_class_hierarchy()
for row in hierarchy:
    child = row['child'].split('/')[-1].split('#')[-1]
    parent = row['parent'].split('/')[-1].split('#')[-1]
    print(f'  {child} ──subClassOf──▶ {parent}')

In [ ]:
# Query 3: Custom SPARQL
sparql = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?class (COUNT(?sub) AS ?subclass_count) WHERE {
    ?sub rdfs:subClassOf ?class .
    ?class a owl:Class .
    FILTER(isURI(?class))
}
GROUP BY ?class
ORDER BY DESC(?subclass_count)
"""

print('\n=== Classes by Number of Subclasses ===')
results = engine.query(sparql)
for r in results:
    name = r['class'].split('/')[-1].split('#')[-1]
    print(f'  {name:<20} subclasses: {r["subclass_count"]}')

## 4. Visualize the Ontology

In [ ]:
from src.visualizer import OntologyVisualizer

viz = OntologyVisualizer('output/test_ontology.owl')

print('Graph summary:', viz.get_summary())

viz.plot_hierarchy(
    figsize=(14, 10),
    title='Animal Ontology — Class Hierarchy',
    save_path='output/ontology_graph.png'
)

In [ ]:
# Export for WebVOWL (online interactive viewer)
viz.export_for_webvowl('output/ontology.ttl')
print('\nUpload output/ontology.ttl to http://www.visualdataweb.de/webvowl/')

---
**Next:** [Notebook 6 — Full Pipeline Demo](06_full_pipeline_demo.ipynb)